In [1188]:
import os,sys
sys.path.insert(1, os.path.join(os.getcwd()  , '..'))

In [1189]:
import importlib, shallowsim as sb
import pandas as pd
import math

In [1190]:
importlib.reload(sb)

<module 'shallowsim' from 'c:\\Users\\陈铂英\\Desktop\\shallowsim\\shallowsim.py'>

In [1191]:
args = sb.ModelArgs.load_from_csv("modelArgs.csv","qwen1.5-MoE") # tinyllama-1.1B qwen1.5-MoE
c = sb.Config()

In [1192]:
print(args.is_moe)
print(args.attention)

True
gqa


In [1193]:
gpu_blackwell = sb.get_gpu_info('./device/gpu_info.csv',print_console=True ) 

| gpu_type   |   sm |   comm_sm |   fp16 |   fp8 |   fp4 |   mem |   mem_bw |   nvlink_bw |   pcie_bw |   gpu_per_node |
|:-----------|-----:|----------:|-------:|------:|------:|------:|---------:|------------:|----------:|---------------:|
| A800       |  108 |        13 |    311 |   311 |     0 |    80 |     2039 |         600 |        50 |              8 |


In [1194]:
detail,summary = sb.prefill_time(args,gpu_blackwell, c, c.seq_len, c.kv_cache_rate, tp=8, dp=1)

In [1195]:
detail

GPU,Layers,A800
MLP_TPN,0.0,0.063319
Attn_TPN,24.0,0.054991
Shared Expert,24.0,0.014284
Combine,24.0,0.067647
Routed Expert,24.0,0.073796
Dispatch,24.0,0.054412


In [1196]:
summary

GPU,A800
Compute,3.433718
Comm,2.929412
Overlapped,0.585882
Sum,5.777248


In [1197]:
detail,summary = sb.prefill_time_pp(args, c, gpu_blackwell, c.kv_cache_rate, tp=1, dp=1, pp=1, num_chunks = 2)

In [1198]:
summary
# serial total: Sum * bs

GPU,A800
Compute,22.284163
Comm,6.635294
Overlapped,1.327059
Sum,27.592398


In [1199]:
# prefill moe

In [1200]:
tp = 2
_, ttft_sum = sb.prefill_time(args, gpu_blackwell, c, c.seq_len, c.kv_cache_rate, tp=tp, dp=2, print_console=False)
print(ttft_sum.apply(lambda x: c.seq_len/tp * (1000/ x)).loc['Sum'].to_markdown(floatfmt=".1f"))

| GPU   |     Sum |
|:------|--------:|
| A800  | 37514.9 |


# Test decode (TP & PP)

In [1201]:
args = sb.ModelArgs.load_from_csv("modelArgs.csv","qwen1.5-MoE") # qwen1.5-MoE tinyllama-1.1B
c = sb.Config()

In [1202]:
print(args.is_moe)
print(args.attention)

True
gqa


In [1203]:
gpu_blackwell_decode = sb.get_gpu_info('./device/gpu_info.csv', decoding_mode=True, print_console=True) 

| gpu_type   |   sm |   comm_sm |   fp16 |   fp8 |   fp4 |   mem |   mem_bw |   nvlink_bw |   pcie_bw |   gpu_per_node |
|:-----------|-----:|----------:|-------:|------:|------:|------:|---------:|------------:|----------:|---------------:|
| A800       |  108 |        13 |    311 |   311 |     0 |    80 |     2039 |         600 |        50 |              8 |


In [1204]:

detail = sb.decode_time(
        args,                     
        gpu_blackwell_decode, 
        c,               
        [1,8,16],           
        c.seq_len,
        c.decode_len,
        gemm_group_per_device=math.ceil(args.n_routed_experts / 8),  
        device_num=8,             
        fp8_combine=False,
        tps_limit=0,
        print_console=True) 


|                   |   Attn_TPN |   MLP_TPN |   SharedExpert |   RoutedExpert |   Dispatch |   Combine |   Overlapped |   TPOT |   Comm_Time_Ratio |     TPS |    Total |
|:------------------|-----------:|----------:|---------------:|---------------:|-----------:|----------:|-------------:|-------:|------------------:|--------:|---------:|
| ('A800', '1', 1)  |      0.053 |     0.019 |          0.011 |          0.081 |      0.010 |     0.010 |        0.247 |  3.756 |             0.066 | 266.230 |  266.230 |
| ('A800', '1', 2)  |      0.041 |     0.025 |          0.011 |          0.081 |      0.010 |     0.010 |        0.247 |  3.482 |             0.071 | 287.206 |  287.206 |
| ('A800', '1', 4)  |      0.028 |     0.020 |          0.011 |          0.081 |      0.010 |     0.010 |        0.247 |  3.165 |             0.078 | 316.000 |  316.000 |
| ('A800', '1', 8)  |      0.022 |     0.017 |          0.011 |          0.081 |      0.010 |     0.010 |        0.247 |  3.006 |             0.0

In [1086]:
result = sb.decode_time_pp(
        args,
        c,                                          
        gpu_blackwell_decode,            
        gemm_group_per_device=math.ceil(args.n_routed_experts / 8),  
        device_num=8,   
        pp=2,micro_bs_num=2,
        tps_limit=0,                                           
        fp8_combine=False,
        print_console=True) 


[Decode · Pipeline-parallel with PP = 2]
|                 |    A800 |    A800 |    A800 |    A800 |     A800 |     A800 |     A800 |     A800 |     A800 |     A800 |     A800 |     A800 |
|:----------------|--------:|--------:|--------:|--------:|---------:|---------:|---------:|---------:|---------:|---------:|---------:|---------:|
| BatchSize       |   1.000 |   1.000 |   1.000 |   1.000 |    8.000 |    8.000 |    8.000 |    8.000 |   16.000 |   16.000 |   16.000 |   16.000 |
| TP              |   1.000 |   2.000 |   4.000 |   8.000 |    1.000 |    2.000 |    4.000 |    8.000 |    1.000 |    2.000 |    4.000 |    8.000 |
| LoadKV          |   0.001 |   0.001 |   0.001 |   0.001 |    0.009 |    0.009 |    0.009 |    0.009 |    0.018 |    0.018 |    0.018 |    0.018 |
| Attn_TP1        |   0.053 |   0.053 |   0.053 |   0.053 |    0.054 |    0.054 |    0.054 |    0.054 |    0.055 |    0.055 |    0.055 |    0.055 |
| Attn_TPN        |   0.053 |   0.041 |   0.028 |   0.022 |    0.054 |

In [1087]:
dfs_o = detail.groupby(['GPU','BatchSize'],as_index=False).apply(lambda t: t[t.Total==t.Total.max()]).sort_values(['Total'],ascending=False).reset_index(drop=True)

dfs_o.style.bar(subset=['TPS','Total'],color='#6495ED')\
      .applymap(sb.gpu_category_color,props=sb.gpu_category_idx(gpu_blackwell_decode),subset=['GPU'])\
      .format(precision=3) 

,GPU,BatchSize,TP,LoadKV,Attn_TP1,Attn_TPN,MLP_TPN,SharedExpert,RoutedExpert,Dispatch,Combine,TPOT,TPS,Total,ComputeTime,CommTime,Overlapped,Comm_Time_Ratio
0,A800,16,8,0.018,0.055,0.022,0.018,0.030,0.160,0.013,0.016,5.876,170.186,2722.972,5.530,0.692,0.346,0.059
1,A800,8,8,0.009,0.054,0.022,0.018,0.020,0.118,0.011,0.013,4.345,230.136,1841.085,4.052,0.586,0.293,0.067
2,A800,1,8,0.001,0.053,0.022,0.017,0.011,0.081,0.010,0.010,3.006,332.676,332.676,2.759,0.493,0.247,0.082


In [1088]:
sb.df_filter(detail,'DGX-B300',0).style\
      .bar(subset=['TPS','Total'],color='#6495ED')\
      .applymap(sb.color_positive_red, subset=['Delta'])\
      .background_gradient(subset=['Comm_Impact'],cmap=sb.cm)\
      .format(precision=3) 

KeyError: "None of [Index(['Delta'], dtype='object')] are in the [columns]"